# AdaptiveSats Regime-Based BTC Accumulation Strategy

**Author:** Raghav Gupta

This notebook implements and evaluates multiple regime-based Bitcoin accumulation strategies against a uniform DCA benchmark.

## Main terms used in the notebook

- **DCA**: Uniform dollar-cost averaging. Every day in a 365-day window gets the same allocation weight.
- **SPD / sats per dollar**: Number of satoshis accumulated per USD spent. Higher is better.
- **Window**: A fixed 365-day evaluation period. Each window receives the same budget.
- **Regime**: A market condition label based on BTC trend, MVRV valuation, and realized-cap versus market-cap growth.
- **Candidate strategy**: A strategy that can be selected inside a regime, such as MVRV, Momentum, SMA, or Composite.
- **Composite strategy**: A weighted combination of on-chain signals using causal rolling z-scores.
- **Causal rolling z-score**: A z-score calculated using only current and past values, not future values.
- **MAX_DCA_MULTIPLE**: The cap that prevents any single day from receiving more than a fixed multiple of normal DCA allocation.


## Cell 1: Imports, data preparation check, and configuration

Set up the notebook: imports, StackSats dataset checks, train/test dates, budget, lookbacks, allocation cap, composite signal list, and candidate strategy names.

In [1]:
# Cell 1: Imports, data preparation check, and configuration
# ============================================================
# This cell imports required libraries, checks whether the prepared
# StackSats BTC analytics dataset exists, prepares it if missing,
# and defines all strategy settings.

import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import subprocess

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

# Optimization import used by the composite strategy.
# If this fails, install scipy in your environment:
# pip install scipy
try:
    from scipy.optimize import minimize
except ImportError as exc:
    raise ImportError(
        "scipy is not installed. Install it using: pip install scipy "
        "or add `scipy` to your environment.yml."
    ) from exc

# ============================================================
# StackSats prepared dataset check
# ============================================================
# pip install stacksats installs the package, but it does not automatically
# create ~/.stacksats/data/bitcoin_analytics.parquet.
# This block prepares the file if it is missing.

btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"

# IMPORTANT:
# Update this path if your brk_metrics.parquet is in a different location.
# If this notebook is inside the notebooks/ folder and data/ is at repo root,
# then ../data/brk_metrics.parquet is usually correct.
raw_brk_path = Path("../data/brk_metrics.parquet")

# Processed long-format BRK metrics used for the composite strategy.
# Expected format: day_utc | metric | value
processed_brk_metrics_path = Path("../data/processed/brk_metrics.parquet")

if not btc_path.exists():
    print(f"Prepared dataset not found at: {btc_path}")
    print("Preparing StackSats analytics dataset...")

    if not raw_brk_path.exists():
        raise FileNotFoundError(
            f"Raw BRK metrics file not found at: {raw_brk_path}. "
            "Please update raw_brk_path to the correct location of brk_metrics.parquet."
        )

    subprocess.run(
        [
            "stacksats",
            "data",
            "prepare",
            "--source",
            str(raw_brk_path),
        ],
        check=True,
    )

if not btc_path.exists():
    raise FileNotFoundError(
        f"Failed to create prepared dataset at {btc_path}."
    )

print(f"Using prepared dataset: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per 365-day window.
TOTAL_BUDGET_USD = 1000.0

# Train and test periods.
TRAIN_START = "2018-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Each evaluation window is 365 days.
WINDOW_SIZE = 365

# StackSats-style exponential decay factor for percentile aggregation.
# 0.9 means each older window receives 90% of the weight of the next newer window.
EXP_DECAY_FACTOR = 0.90


# Lookbacks used for momentum, SMA, drawdown, and regime classification.
MOMENTUM_LOOKBACK = 45
SMA_LOOKBACK = 180
DRAWDOWN_LOOKBACK = 180
REGIME_LOOKBACK = 180

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Maximum final daily allocation relative to uniform DCA.
# Example: for a 365-day window, DCA weight = 1/365 = 0.0027397.
# With MAX_DCA_MULTIPLE = 15, no day can receive more than:
# 15 * 1/365 = 0.041096, or about 4.11% of the window budget.
# Fixed final daily allocation cap.
# No day can receive more than MAX_DCA_MULTIPLE times the uniform DCA allocation.
MAX_DCA_MULTIPLE = 15.0

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Composite on-chain strategy settings.
# This strategy uses extra BRK metrics from:
# ../data/processed/brk_metrics.parquet
#
# The file is expected to be long-format:
# day_utc | metric | value
#
# Nelder-Mead starts from equal weights and optimizes signal weights + gamma
# using the training period only.
COMPOSITE_ROLLING_WINDOW = 365
COMPOSITE_SIGNAL_STRENGTH = 1.25
COMPOSITE_GAMMA = 1.00

COMPOSITE_SIGNAL_COLS = [
    "mvrv",
    "sopr_7d_ema",
    "reserve_risk",
    "greed_index",
    "puell_multiple",
    "lth_nupl",
    "sell_side_risk_ratio_7d_ema",
    "net_unrealized_pnl_rel_to_market_cap",
]

COMPOSITE_Z_COLS = [f"z_{col}" for col in COMPOSITE_SIGNAL_COLS]

COMPOSITE_SIGNAL_WEIGHTS = {
    col: 1.0 / len(COMPOSITE_SIGNAL_COLS)
    for col in COMPOSITE_SIGNAL_COLS
}

# Nelder-Mead optimization settings for the composite strategy.
COMPOSITE_OPT_MAXITER = 600
COMPOSITE_OPT_MAXFEV = 1200

# Gamma controls how aggressively positive cheapness is amplified.
# It is clipped during optimization to reduce overfitting.
COMPOSITE_GAMMA_MIN = 0.25
COMPOSITE_GAMMA_MAX = 3

# Fallback strategy used if a regime appears in test but was not seen in training.
FALLBACK_STRATEGY =  "composite_weight"

# Candidate strategies used in regime selection.
# DCA is benchmark only and is not selected as a candidate here.
# ML target-based strategy has been removed to avoid train/test label-boundary leakage.
# Standalone StackSats MVRV, Momentum, SMA, and Composite remain as candidates.
CANDIDATE_COLS = [
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
    f"sma_{SMA_LOOKBACK}d_weight",
    "composite_weight",
]


# ============================================================


Using prepared dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet


## Cell 2: Helper functions

Reusable utility functions for labeling results, normalizing weights, computing rolling z-scores, calculating exponential-decay percentiles, and summarizing sats-per-dollar performance.

In [2]:
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """
    if pct_diff > tolerance:
        return "better"
    if pct_diff < -tolerance:
        return "worse"
    return "tie"


def format_arrow_text(extra_spd, improvement_pct):
    """
    Create chart annotation text for positive or negative SPD improvement.

    Parameters
    ----------
    extra_spd : float
        Extra sats per dollar compared with DCA.
    improvement_pct : float
        Percentage improvement compared with DCA.

    Returns
    -------
    tuple[str, str]
        Formatted text and color name.
    """
    if extra_spd >= 0:
        return f"▲ +{extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "green"
    return f"▼ {extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "red"


def trim_full_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE):
    """
    Keep only complete fixed-length windows.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe sorted by date.
    window_size : int
        Number of rows per evaluation window.

    Returns
    -------
    tuple[pd.DataFrame, int, int]
        Trimmed dataframe, number of complete windows, and number of dropped rows.
    """
    n_full = len(df) // window_size
    n_eval = n_full * window_size
    remainder = len(df) - n_eval
    return df.iloc[:n_eval].copy(), n_full, remainder


def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    clean_signal = np.maximum(clean_signal, signal_floor)

    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    return clean_signal / clean_signal.sum()


def rolling_zscore(series: pd.Series, window: int = COMPOSITE_ROLLING_WINDOW):
    """
    Compute a causal rolling z-score.

    Each row uses only current/trailing historical values.
    No future values are used.
    """
    rolling_mean = series.rolling(window=window, min_periods=120).mean()
    rolling_std = series.rolling(window=window, min_periods=120).std()
    return (series - rolling_mean) / (rolling_std + 1e-8)


def compute_stack_sats_exp_decay_average(
    percentile_values,
    decay_factor=EXP_DECAY_FACTOR,
):
    """
    Calculate StackSats-style exponentially decayed average percentile.

    Parameters
    ----------
    percentile_values : array-like
        Chronologically ordered percentile values.
        Older values should come first and newer values should come last.
    decay_factor : float
        Exponential decay factor.

    Returns
    -------
    float
        Exponentially decayed average percentile.

        exp_weights = 0.9 ** np.arange(N - 1, -1, -1)
        exp_weights /= exp_weights.sum()
        exp_avg_pct = (percentile_values * exp_weights).sum()

    This gives the newest window the largest weight and older windows
    gradually lower weights.
    """
    values = np.asarray(percentile_values, dtype=float)

    if len(values) == 0:
        return np.nan

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    n = len(values)

    exp_weights = decay_factor ** np.arange(n - 1, -1, -1)
    exp_weights = exp_weights / exp_weights.sum()

    return float((values * exp_weights).sum())


def add_spd_percentiles_to_window_summary(window_summary_df):
    """
    Add dynamic and uniform SPD percentile columns.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary with strategy_spd and dca_spd.

    Returns
    -------
    pd.DataFrame
        Window summary with:
        - dynamic_percentile
        - uniform_percentile

    Notes
    -----
    Here, percentiles are calculated across the available 365-day windows
    in the current period.
    """
    out = window_summary_df.copy().sort_values("start_date").reset_index(drop=True)

    out["dynamic_percentile"] = (
        out["strategy_spd"]
        .rank(method="average", pct=True)
        * 100.0
    )

    out["uniform_percentile"] = (
        out["dca_spd"]
        .rank(method="average", pct=True)
        * 100.0
    )

    return out


def calculate_stack_sats_exp_decay_metrics(window_summary_df):
    """
    Calculate exp-decay percentile metrics for a window summary.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary with dynamic_percentile and uniform_percentile.

    Returns
    -------
    dict
        Dictionary with exp_decay_percentile and uniform_exp_decay_percentile.
    """
    ordered = window_summary_df.copy().sort_values("start_date").reset_index(drop=True)

    exp_decay_percentile = compute_stack_sats_exp_decay_average(
        ordered["dynamic_percentile"].to_numpy(),
        decay_factor=EXP_DECAY_FACTOR,
    )

    uniform_exp_decay_percentile = compute_stack_sats_exp_decay_average(
        ordered["uniform_percentile"].to_numpy(),
        decay_factor=EXP_DECAY_FACTOR,
    )

    return {
        "exp_decay_percentile": exp_decay_percentile,
        "uniform_exp_decay_percentile": uniform_exp_decay_percentile,
    }




def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """
    n_windows = len(window_summary_df)

    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    spd_ratio = strategy_spd_sum / dca_spd_sum
    improvement_pct = (spd_ratio - 1.0) * 100.0

    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()
    extra_sats = strategy_sats - dca_sats

    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    if {"dynamic_percentile", "uniform_percentile"}.issubset(window_summary_df.columns):
        exp_decay_metrics = calculate_stack_sats_exp_decay_metrics(window_summary_df)
    else:
        exp_decay_metrics = {
            "exp_decay_percentile": np.nan,
            "uniform_exp_decay_percentile": np.nan,
        }

    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,
        "exp_decay_percentile": exp_decay_metrics.get("exp_decay_percentile", np.nan),
        "uniform_exp_decay_percentile": exp_decay_metrics.get("uniform_exp_decay_percentile", np.nan),

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


def build_log_tick_values_and_text():
    """
    Build custom y-axis tick values and labels for the BTC log-price chart.

    Returns
    -------
    tuple[list[int], list[str]]
        Tick values and corresponding display labels.
    """
    tickvals = [
        3000, 4000, 5000, 6000, 7000, 8000, 9000,
        10000,
        20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000,
        100000,
    ]

    ticktext = [
        "3", "4", "5", "6", "7", "8", "9",
        "10k",
        "2", "3", "4", "5", "6", "7", "8", "9",
        "100k",
    ]

    return tickvals, ticktext


# ============================================================


## Cell 3: Load BTC data

Load the prepared Bitcoin analytics dataset and verify that all required price and on-chain columns exist before strategy logic runs.

In [3]:
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(btc_path)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)


# ============================================================


Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


## Cell 4: StackSats strategy export setup

Initialize the StackSats runner and define a cached exporter for built-in MVRV and Momentum daily allocation weights.

In [4]:
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner.
# It exports daily weights from built-in StackSats MVRV and Momentum strategies.

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weights_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one 365-day window.

    Parameters
    ----------
    strategy_key : str
        Key identifying which StackSats strategy to run.
    window_df : pd.DataFrame
        Current 365-day window dataframe.
    full_btc_df : pl.DataFrame
        Full BTC analytics dataframe in Polars format.

    Returns
    -------
    np.ndarray
        Normalized daily weights for the selected StackSats strategy.
    """
    window_start = pd.to_datetime(window_df["date"].min()).strftime("%Y-%m-%d")
    window_end = pd.to_datetime(window_df["date"].max()).strftime("%Y-%m-%d")

    cache_key = (strategy_key, window_start, window_end)

    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    config = ExportConfig(
        range_start=window_start,
        range_end=window_end,
    )

    window_btc_df = (
        full_btc_df
        .filter(
            (pl.col("date") >= pd.to_datetime(window_start)) &
            (pl.col("date") <= pd.to_datetime(window_end)) &
            pl.col("price_usd").is_not_null()
        )
        .sort("date")
    )

    if window_btc_df.is_empty():
        raise ValueError(f"No BTC data available for {window_start} to {window_end}")

    strategy = stacksats_strategy_objects[strategy_key]

    export_obj = runner.export(
        strategy,
        config,
        btc_df=window_btc_df,
    )

    weights = export_obj.to_dataframe()

    if not isinstance(weights, pl.DataFrame):
        weights = pl.from_pandas(weights)

    weights = weights.with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime),
        pl.col("date").cast(pl.Datetime),
    ])

    latest_end = weights.select(pl.col("end_date").max()).item()

    one_window = (
        weights
        .filter(pl.col("end_date") == latest_end)
        .sort("date")
        .select(["date", "weight"])
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    one_window["date"] = pd.to_datetime(one_window["date"])

    merged = (
        window_df[["date"]]
        .merge(one_window, on="date", how="left")
        .sort_values("date")
        .reset_index(drop=True)
    )

    if merged["raw_weight"].isna().any():
        missing_dates = (
            merged.loc[merged["raw_weight"].isna(), "date"]
            .dt.strftime("%Y-%m-%d")
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"Missing StackSats weights for {strategy_key} "
            f"from {window_start} to {window_end}. "
            f"Example missing dates: {missing_dates}"
        )

    # This function returns the original StackSats normalized weights.
    # The final regime-selected weights are capped later in Cell 12.
    final_weights = build_simple_normalized_weights(
        merged["raw_weight"].values
    )

    _export_cache[cache_key] = final_weights.copy()

    return final_weights


# ============================================================
